# Vehicles YOLO: download, EDA, training, evaluation, and inference

Notebook ini dibuat untuk dataset Vehicles dari Roboflow 100. Urutan kerja:

1. Install dependency dan download dataset format YOLOv8.
2. Validasi struktur, visualisasi anotasi, dan EDA.
3. Train baseline di Colab secara manual.
4. Evaluasi validasi/test, confusion matrix, PR curve, dan error analysis.
5. Evaluasi model hasil penggabungan kelas dan jalankan inference.
6. Uji model final pada gambar dan video, termasuk FPS.
7. Tulis report ke `reports/report.md`.

API key tidak disimpan di notebook. Gunakan environment variable `ROBOFLOW_API_KEY`; jika tidak tersedia, notebook akan meminta key dengan input tersembunyi. Key yang pernah tertulis di chat sebaiknya segera di-revoke dan dibuat ulang.

Jalankan cell training dan improvement hanya di Colab dengan GPU. Cell evaluasi setelah itu memakai `weights/best.pt` atau path model yang dipilih.

In [ ]:
%pip install -q ultralytics roboflow opencv-python matplotlib seaborn pandas pyyaml scikit-learn pillow

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import random
import shutil
import time
from collections import Counter, defaultdict
from getpass import getpass

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

from IPython.display import Image as IPImage, display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'vehicles'
RUNS_ROOT = PROJECT_ROOT / 'runs'
WEIGHTS_ROOT = PROJECT_ROOT / 'weights'
REPORTS_ROOT = PROJECT_ROOT / 'reports'
for directory in (DATASET_ROOT, RUNS_ROOT, WEIGHTS_ROOT, REPORTS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Download dataset

Format `yolov8` dipakai karena langsung kompatibel dengan Ultralytics dan menghasilkan `data.yaml` beserta folder `images/labels`. Dataset disimpan di `datasets/vehicles/` agar path konsisten untuk EDA dan training.

In [ ]:
ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass('Masukkan Roboflow API key: ')
if not ROBOFLOW_API_KEY:
    raise ValueError('ROBOFLOW_API_KEY kosong.')

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('muhammad-faris').project('vehicles-q0x2v-kpycp')
version = project.version(1)
dataset = version.download('yolov8')
DATASET_ROOT = Path(dataset.location)
DATA_YAML = DATASET_ROOT / 'data.yaml'
print(f'Dataset location: {DATASET_ROOT}')
print(f'Data YAML: {DATA_YAML}')
assert DATA_YAML.exists(), f'data.yaml tidak ditemukan: {DATA_YAML}'

In [ ]:
DATASET_ROOT = Path(dataset.location)
DATA_YAML = DATASET_ROOT / 'data.yaml'

with DATA_YAML.open() as file:
    DATA_CONFIG = yaml.safe_load(file)

raw_names = DATA_CONFIG.get('names', [])
if isinstance(raw_names, dict):
    CLASS_NAMES = [raw_names[key] for key in sorted(raw_names, key=lambda value: int(value))]
else:
    CLASS_NAMES = list(raw_names)

def image_paths(split):
    key = 'val' if split == 'valid' else split
    value = DATA_CONFIG.get(key)
    if value is None:
        return []
    path = Path(value)
    if not path.is_absolute():
        path = DATASET_ROOT / path
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
    if path.is_dir():
        return sorted(item for item in path.rglob('*') if item.suffix.lower() in extensions)
    if path.is_file() and path.suffix.lower() in extensions:
        return [path]
    return sorted(item for item in path.parent.glob(path.name) if item.suffix.lower() in extensions)

SPLIT_IMAGES = {split: image_paths(split) for split in ('train', 'valid', 'test')}
print('Classes:', CLASS_NAMES)
print('Jumlah kelas:', len(CLASS_NAMES))
for split, paths in SPLIT_IMAGES.items():
    print(f'{split:>5}: {len(paths):>4} gambar')

assert CLASS_NAMES, 'Daftar kelas kosong.'
assert SPLIT_IMAGES['train'], 'Gambar train tidak ditemukan.'

## 2. Verifikasi anotasi dan visualisasi

Visualisasi dilakukan sebelum training supaya anotasi yang salah, koordinat di luar gambar, atau kelas yang tidak sesuai bisa ditemukan lebih awal.

In [ ]:
def label_path_for_image(image_path):
    image_path = Path(image_path)
    parts = list(image_path.parts)
    if 'images' in parts:
        parts[parts.index('images')] = 'labels'
        return Path(*parts).with_suffix('.txt')
    return image_path.with_suffix('.txt')

def load_yolo_labels(image_path):
    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        return []
    labels = []
    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue
        try:
            numbers = [float(value) for value in values]
        except ValueError:
            continue
        if not numbers[0].is_integer():
            continue
        labels.append((int(numbers[0]), *numbers[1:]))
    return labels

def plot_samples(split='train', count=12):
    paths = SPLIT_IMAGES[split]
    if not paths:
        print(f'Tidak ada gambar pada split {split}.')
        return
    samples = random.Random(SEED).sample(paths, min(count, len(paths)))
    columns = 3
    rows = int(np.ceil(len(samples) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(16, 5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, image_path in zip(axes, samples):
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        for class_id, x_center, y_center, box_width, box_height in load_yolo_labels(image_path):
            x1 = int((x_center - box_width / 2) * width)
            y1 = int((y_center - box_height / 2) * height)
            x2 = int((x_center + box_width / 2) * width)
            y2 = int((y_center + box_height / 2) * height)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 220, 0), 2)
            label = CLASS_NAMES[class_id] if 0 <= class_id < len(CLASS_NAMES) else f'class_{class_id}'
            cv2.putText(image, label, (max(0, x1), max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 60, 0), 2)
        axis.imshow(image)
        axis.set_title(image_path.name)
        axis.axis('off')
    for axis in axes[len(samples):]:
        axis.axis('off')
    figure.suptitle(f'Sampel anotasi: {split}', fontsize=16)
    figure.tight_layout()
    plt.show()

plot_samples('train', count=12)

In [ ]:
def validate_labels(split):
    invalid = []
    missing = []
    empty = []
    for image_path in SPLIT_IMAGES[split]:
        label_path = label_path_for_image(image_path)
        if not label_path.exists():
            missing.append(str(image_path))
            continue
        lines = [line for line in label_path.read_text().splitlines() if line.strip()]
        if not lines:
            empty.append(str(image_path))
        for line_number, line in enumerate(lines, start=1):
            values = line.split()
            try:
                numbers = [float(value) for value in values]
                valid = (len(numbers) == 5 and numbers[0].is_integer() and 0 <= int(numbers[0]) < len(CLASS_NAMES) and all(0 <= value <= 1 for value in numbers[1:]))
            except ValueError:
                valid = False
            if not valid:
                invalid.append((str(label_path), line_number, line))
    return {'missing': missing, 'empty': empty, 'invalid': invalid}

validation = {split: validate_labels(split) for split in SPLIT_IMAGES}
for split, result in validation.items():
    print(f'{split}: missing={len(result["missing"])}, empty={len(result["empty"])}, invalid={len(result["invalid"])}')
    if result['invalid']:
        print('Contoh label invalid:', result['invalid'][:3])

## 3. EDA singkat

EDA memakai anotasi train untuk memahami ketidakseimbangan kelas dan ukuran objek. Pemeriksaan duplikat mencakup duplikat exact dan kemiripan perceptual hash, termasuk kemungkinan kebocoran antar split.

In [ ]:
def size_bucket(area_ratio):
    if area_ratio < 0.01:
        return 'small (<1%)'
    if area_ratio < 0.10:
        return 'medium (1-10%)'
    return 'large (>=10%)'

eda_rows = []
for split, paths in SPLIT_IMAGES.items():
    for image_path in paths:
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        height, width = image.shape[:2]
        for class_id, x_center, y_center, box_width, box_height in load_yolo_labels(image_path):
            area_ratio = box_width * box_height
            eda_rows.append({
                'split': split,
                'image': str(image_path),
                'class_id': class_id,
                'class_name': CLASS_NAMES[class_id] if 0 <= class_id < len(CLASS_NAMES) else f'class_{class_id}',
                'width_px': width,
                'height_px': height,
                'box_width_ratio': box_width,
                'box_height_ratio': box_height,
                'area_ratio': area_ratio,
                'size': size_bucket(area_ratio),
            })

eda_df = pd.DataFrame(eda_rows)
if eda_df.empty:
    raise ValueError('Tidak ada instance anotasi yang terbaca.')

print('Distribusi instance per kelas:')
display(eda_df.groupby(['split', 'class_name']).size().unstack(fill_value=0))
print('Distribusi ukuran objek:')
display(eda_df.groupby(['split', 'size']).size().unstack(fill_value=0))

figure, axes = plt.subplots(1, 2, figsize=(17, 5))
class_counts = eda_df.groupby(['split', 'class_name']).size().unstack(fill_value=0)
class_counts.plot(kind='bar', ax=axes[0])
axes[0].set_title('Instance per kelas dan split')
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Jumlah instance')
axes[0].tick_params(axis='x', rotation=0)
sns.histplot(data=eda_df[eda_df['split'] == 'train'], x='area_ratio', hue='class_name', bins=30, element='step', stat='count', ax=axes[1])
axes[1].set_title('Distribusi area bounding box train')
axes[1].set_xlabel('Area box / area gambar')
figure.tight_layout()
plt.show()

In [ ]:
def average_hash(image_path, hash_size=8):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    image = cv2.resize(image, (hash_size, hash_size), interpolation=cv2.INTER_AREA)
    return ''.join('1' if pixel >= image.mean() else '0' for pixel in image.ravel())

duplicate_rows = []
for split, paths in SPLIT_IMAGES.items():
    for image_path in paths:
        content = image_path.read_bytes()
        duplicate_rows.append({
            'split': split,
            'image': str(image_path),
            'sha256': hashlib.sha256(content).hexdigest(),
            'average_hash': average_hash(image_path),
        })
duplicate_df = pd.DataFrame(duplicate_rows)

def duplicate_groups(column):
    groups = duplicate_df.groupby(column).agg(count=('image', 'size'), splits=('split', lambda values: ','.join(sorted(set(values)))), images=('image', list))
    return groups[groups['count'] > 1].sort_values('count', ascending=False)

exact_duplicates = duplicate_groups('sha256')
similar_duplicates = duplicate_groups('average_hash')
print(f'Exact duplicate groups: {len(exact_duplicates)}')
if not exact_duplicates.empty:
    display(exact_duplicates.head(10))
print(f'Perceptual duplicate groups: {len(similar_duplicates)}')
if not similar_duplicates.empty:
    display(similar_duplicates.head(10))

cross_split_hashes = duplicate_df.groupby('sha256')['split'].nunique()
leakage_candidates = cross_split_hashes[cross_split_hashes > 1]
print(f'Exact duplicate candidates across split: {len(leakage_candidates)}')
if len(leakage_candidates):
    display(duplicate_df[duplicate_df['sha256'].isin(leakage_candidates.index)])

## 4. Training model yang diperbaiki

Dataset asli memiliki banyak kelas dengan jumlah contoh sangat kecil. Untuk model realtime yang stabil, kelas tersebut digabung menjadi `car`, `bus`, dan `truck`. Cell ini otomatis menyiapkan dataset hasil penggabungan lalu menjalankan satu training `yolov8s`.

Early stopping tetap aktif dengan patience 30. Jangan menjalankan eksperimen model kedua sebelum melihat hasil model ini.

In [ ]:
import ultralytics
from ultralytics import YOLO

MERGED_DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'vehicles_merged'
MERGED_DATA_YAML = MERGED_DATASET_ROOT / 'data.yaml'
MERGED_CLASSES = ['car', 'bus', 'truck']

def merged_class(class_name):
    name = class_name.lower()
    if 'car' in name:
        return 0
    if 'bus' in name:
        return 1
    return 2

def prepare_merged_dataset():
    if MERGED_DATA_YAML.exists():
        return
    for split in ('train', 'valid', 'test'):
        source_key = 'valid' if (DATASET_ROOT / 'valid').exists() and split == 'valid' else ('val' if split == 'valid' else split)
        source_images = DATASET_ROOT / source_key / 'images'
        source_labels = DATASET_ROOT / source_key / 'labels'
        target_images = MERGED_DATASET_ROOT / split / 'images'
        target_labels = MERGED_DATASET_ROOT / split / 'labels'
        target_images.mkdir(parents=True, exist_ok=True)
        target_labels.mkdir(parents=True, exist_ok=True)
        for image_path in source_images.glob('*'):
            if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
                continue
            shutil.copy2(image_path, target_images / image_path.name)
            source_label = source_labels / f'{image_path.stem}.txt'
            output = []
            if source_label.exists():
                for line in source_label.read_text().splitlines():
                    values = line.split()
                    if len(values) != 5:
                        continue
                    class_id = int(float(values[0]))
                    if 0 <= class_id < len(CLASS_NAMES):
                        output.append(f'{merged_class(CLASS_NAMES[class_id])} ' + ' '.join(values[1:]))
            (target_labels / f'{image_path.stem}.txt').write_text('\n'.join(output) + ('\n' if output else ''))
    MERGED_DATA_YAML.write_text(yaml.safe_dump({
        'path': str(MERGED_DATASET_ROOT),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(MERGED_CLASSES),
        'names': MERGED_CLASSES,
    }, sort_keys=False))

prepare_merged_dataset()
DATA_YAML = MERGED_DATA_YAML
CLASS_NAMES = MERGED_CLASSES
SPLIT_IMAGES = {split: image_paths(split) for split in ('train', 'valid', 'test')}
print(f'Merged dataset: {DATA_YAML}')
print({split: len(paths) for split, paths in SPLIT_IMAGES.items()})

def train_model(model_name='yolov8s.pt', run_name='vehicles_yolov8s_merged', batch=16, epochs=100):
    model = YOLO(model_name)
    started = time.perf_counter()
    results = model.train(
        data=str(DATA_YAML), imgsz=640, epochs=epochs, batch=batch,
        patience=30, device=DEVICE, project=str(RUNS_ROOT),
        name=run_name, exist_ok=True, plots=True, pretrained=True,
    )
    elapsed = time.perf_counter() - started
    run_dir = Path(getattr(results, 'save_dir', RUNS_ROOT / run_name))
    best_path = run_dir / 'weights' / 'best.pt'
    info = {'model_name': model_name, 'run_name': run_name, 'elapsed_seconds': elapsed,
            'device': str(DEVICE), 'ultralytics': ultralytics.__version__,
            'imgsz': 640, 'epochs': epochs, 'batch': batch, 'patience': 30,
            'classes': MERGED_CLASSES}
    (run_dir / 'training_info.json').write_text(json.dumps(info, indent=2))
    print(f'Training selesai dalam {elapsed / 60:.1f} menit')
    print(f'Best weights: {best_path}')
    return model, run_dir, best_path, info

# Satu-satunya cell training yang perlu dijalankan.
baseline_model, baseline_run_dir, BASELINE_BEST, BASELINE_INFO = train_model(
    batch=16, epochs=100
)

## 5. Helper evaluasi

Evaluasi memakai confidence rendah (`conf=0.001`) supaya PR curve tidak kehilangan kandidat prediksi. Model dipilih berdasarkan `valid`; `test` dipakai sebagai evaluasi final setelah pilihan model ditetapkan.

In [ ]:
def _as_array(value):
    try:
        return np.asarray(value, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return np.array([], dtype=float)

def per_class_metric(box_metrics, attribute):
    values = _as_array(getattr(box_metrics, attribute, []))
    output = np.full(len(CLASS_NAMES), np.nan)
    if len(values) == len(CLASS_NAMES):
        return values
    indices = _as_array(getattr(box_metrics, 'ap_class_index', []))
    for class_index, value in zip(indices.astype(int), values):
        if 0 <= class_index < len(output):
            output[class_index] = value
    return output

def per_class_ap(box_metrics, column):
    output = np.full(len(CLASS_NAMES), np.nan)
    all_ap = np.asarray(getattr(box_metrics, 'all_ap', []), dtype=float)
    indices = _as_array(getattr(box_metrics, 'ap_class_index', []))
    if all_ap.ndim == 2 and all_ap.shape[1] > column:
        for class_index, value in zip(indices.astype(int), all_ap[:, column]):
            if 0 <= class_index < len(output):
                output[class_index] = value
    return output

def metrics_table(metrics, model_label, split):
    box = metrics.box
    precision = per_class_metric(box, 'p')
    recall = per_class_metric(box, 'r')
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
    return pd.DataFrame({
        'model': model_label,
        'split': split,
        'class': CLASS_NAMES,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mAP50': per_class_ap(box, 0),
        'mAP50-95': per_class_ap(box, 0 if np.asarray(getattr(box, 'all_ap', [])).ndim < 2 else np.asarray(box.all_ap).shape[1] - 1),
    })

def overall_metrics(metrics, model_label, split):
    box = metrics.box
    precision = float(box.mp)
    recall = float(box.mr)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        'model': model_label,
        'split': split,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mAP50': float(box.map50),
        'mAP50-95': float(box.map),
    }

def evaluate_model(model_path, split, model_label, run_name):
    split_argument = 'val' if split == 'valid' else split
    model = YOLO(str(model_path))
    started = time.perf_counter()
    metrics = model.val(
        data=str(DATA_YAML),
        split=split_argument,
        imgsz=640,
        batch=16,
        conf=0.001,
        plots=True,
        project=str(RUNS_ROOT / 'evaluation'),
        name=run_name,
        exist_ok=True,
        device=DEVICE,
    )
    elapsed = time.perf_counter() - started
    save_dir = Path(getattr(metrics, 'save_dir', RUNS_ROOT / 'evaluation' / run_name))
    overall = overall_metrics(metrics, model_label, split)
    per_class = metrics_table(metrics, model_label, split)
    print(f'{model_label} / {split} selesai dalam {elapsed:.1f} detik')
    print(pd.DataFrame([overall]).to_string(index=False))
    display(per_class.round(4))
    return model, metrics, save_dir, overall, per_class

def show_evaluation_plots(save_dir):
    files = ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']
    for filename in files:
        path = save_dir / filename
        if path.exists():
            print(filename)
            display(IPImage(filename=str(path)))
        else:
            print(f'Tidak ditemukan: {path}')

## 6. Evaluasi model hasil training

Cell ini mengevaluasi baseline pada `valid`. Jalankan juga pada `test` hanya untuk pencatatan awal; keputusan model tetap dibuat dari `valid` agar test tidak menjadi bagian dari tuning. Confusion matrix dan PR curve dibuat oleh Ultralytics di folder evaluasi.

In [ ]:
if 'BASELINE_BEST' not in globals() or not Path(BASELINE_BEST).exists():
    print('Baseline best.pt belum ada. Jalankan cell training terlebih dahulu.')
else:
    baseline_valid_model, baseline_valid_metrics, baseline_valid_dir, baseline_valid_overall, baseline_valid_per_class = evaluate_model(
        BASELINE_BEST, 'valid', 'merged_yolov8s', 'merged_valid'
    )
    show_evaluation_plots(baseline_valid_dir)

## 7. Evaluasi model hasil training

Model hasil penggabungan kelas langsung dievaluasi. Jangan menjalankan model kedua sebelum model ini diperiksa.

In [ ]:
# Tidak ada eksperimen model kedua: kuota Colab dipakai untuk model yang datanya sudah diperbaiki.
print('Training utama selesai. Lanjutkan ke evaluasi validasi.')

In [ ]:
valid_results = []
if 'baseline_valid_overall' in globals():
    valid_results.append(baseline_valid_overall)

if valid_results:
    comparison_df = pd.DataFrame(valid_results).set_index('model')
    display((comparison_df * 100).round(2).rename(columns={'precision': 'precision_%', 'recall': 'recall_%', 'f1': 'f1_%', 'mAP50': 'mAP50_%', 'mAP50-95': 'mAP50-95_%'}))
    print('Pilih model berdasarkan validasi dan FPS, bukan mAP saja.')
else:
    print('Belum ada hasil validasi untuk dibandingkan.')

## 8. Pilih model final dan evaluasi test

Setelah membandingkan hasil validasi, salin bobot pilihan ke `weights/best.pt`, atau ubah `FINAL_MODEL_PATH` ke bobot run yang dipilih. Di Colab, upload `best.pt` melalui cell berikut jika bobot akan dibawa dari sesi training. Test dijalankan sekali sebagai angka final yang tidak dipakai untuk tuning.

In [ ]:
# Jalankan cell ini di Colab setelah best.pt tersedia.
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        uploaded_name = next(iter(uploaded))
        shutil.copy2(uploaded_name, WEIGHTS_ROOT / 'best.pt')
        print(f'Disalin ke {WEIGHTS_ROOT / "best.pt"}')
except ImportError:
    print('Bukan lingkungan Colab. Letakkan best.pt di weights/best.pt secara manual.')

In [ ]:
FINAL_MODEL_PATH = WEIGHTS_ROOT / 'best.pt'
if not FINAL_MODEL_PATH.exists():
    if 'BASELINE_BEST' in globals() and Path(BASELINE_BEST).exists():
        FINAL_MODEL_PATH = Path(BASELINE_BEST)

if not FINAL_MODEL_PATH.exists():
    print('Model final belum tersedia. Upload atau set FINAL_MODEL_PATH ke file best.pt.')
else:
    final_model, final_test_metrics, final_test_dir, final_test_overall, final_test_per_class = evaluate_model(
        FINAL_MODEL_PATH, 'test', 'final_model', 'final_test'
    )
    show_evaluation_plots(final_test_dir)

In [ ]:
def top_confusions(metrics, limit=8):
    confusion = getattr(metrics, 'confusion_matrix', None)
    matrix = np.asarray(getattr(confusion, 'matrix', []), dtype=float)
    if matrix.ndim != 2:
        return pd.DataFrame()
    matrix = matrix[:len(CLASS_NAMES), :len(CLASS_NAMES)]
    rows = []
    for true_id in range(len(CLASS_NAMES)):
        for predicted_id in range(len(CLASS_NAMES)):
            if true_id != predicted_id and matrix[true_id, predicted_id] > 0:
                rows.append({'true_class': CLASS_NAMES[true_id], 'predicted_class': CLASS_NAMES[predicted_id], 'count': matrix[true_id, predicted_id]})
    return pd.DataFrame(rows).sort_values('count', ascending=False).head(limit) if rows else pd.DataFrame()

if 'final_test_metrics' in globals():
    confusion_pairs = top_confusions(final_test_metrics)
    if confusion_pairs.empty:
        print('Tidak ada pasangan confusion off-diagonal yang terbaca.')
    else:
        display(confusion_pairs)
        for row in confusion_pairs.itertuples():
            print(f"{row.true_class} -> {row.predicted_class}: {int(row.count)} kasus; kemungkinan penyebab: bentuk/ukuran visual mirip, occlusion, atau objek terlalu kecil.")
else:
    confusion_pairs = pd.DataFrame()
    print('Evaluasi test belum dijalankan.')

### Interpretasi confusion matrix dan PR curve

Lihat baris/kolom dengan nilai terbesar pada confusion matrix. Pertukaran `bus-l-`, `bus-s-`, dan `big bus` perlu diperiksa pada gambar asli karena bentuk, jarak, dan resolusi objek dapat sangat mirip. PR curve yang rendah atau turun cepat menunjukkan confidence model belum stabil pada kelas tersebut. Jangan menyimpulkan penyebab sebelum melihat sampel error di bawah.

In [ ]:
def xywhn_to_xyxy(label, width, height):
    class_id, x_center, y_center, box_width, box_height = label
    return np.array([
        (x_center - box_width / 2) * width,
        (y_center - box_height / 2) * height,
        (x_center + box_width / 2) * width,
        (y_center + box_height / 2) * height,
    ], dtype=float)

def box_iou(first, second):
    x1 = max(first[0], second[0])
    y1 = max(first[1], second[1])
    x2 = min(first[2], second[2])
    y2 = min(first[3], second[3])
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    first_area = max(0.0, first[2] - first[0]) * max(0.0, first[3] - first[1])
    second_area = max(0.0, second[2] - second[0]) * max(0.0, second[3] - second[1])
    union = first_area + second_area - intersection
    return intersection / union if union else 0.0

def prediction_arrays(result):
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4)), np.array([], dtype=int), np.array([], dtype=float)
    boxes = result.boxes.xyxy.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confidence = result.boxes.conf.cpu().numpy()
    return boxes, classes, confidence

def analyze_error(image_path, model, iou_threshold=0.5):
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    ground_truth = [(class_id, xywhn_to_xyxy(label, width, height)) for label in load_yolo_labels(image_path) for class_id in [label[0]]]
    result = model.predict(source=str(image_path), imgsz=640, conf=0.001, device=DEVICE, verbose=False)[0]
    predicted_boxes, predicted_classes, predicted_confidence = prediction_arrays(result)
    used_predictions = set()
    matched = []
    false_negative = 0
    for true_class, true_box in ground_truth:
        candidates = [(box_iou(true_box, box), index) for index, (box, class_id) in enumerate(zip(predicted_boxes, predicted_classes)) if index not in used_predictions and class_id == true_class]
        if not candidates or max(candidates)[0] < iou_threshold:
            false_negative += 1
            continue
        iou, prediction_index = max(candidates)
        used_predictions.add(prediction_index)
        matched.append((true_class, iou, predicted_confidence[prediction_index]))
    false_positive = len(predicted_boxes) - len(used_predictions)
    low_confidence = sum(confidence < 0.5 for _, _, confidence in matched)
    return {
        'image_path': str(image_path),
        'result': result,
        'ground_truth': ground_truth,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'low_confidence_matches': low_confidence,
        'matched': matched,
        'error_score': 3 * false_positive + 3 * false_negative + low_confidence,
    }

if 'final_model' in globals() and SPLIT_IMAGES['test']:
    error_items = [analyze_error(path, final_model) for path in SPLIT_IMAGES['test']]
    error_df = pd.DataFrame([{key: value for key, value in item.items() if key not in {'result', 'ground_truth', 'matched'}} for item in error_items]).sort_values('error_score', ascending=False)
    display(error_df.head(10))
else:
    error_items = []
    error_df = pd.DataFrame()
    print('Model final atau gambar test belum tersedia.')

In [ ]:
def display_error_examples(items, count=8):
    selected = sorted(items, key=lambda item: item['error_score'], reverse=True)[:count]
    if not selected:
        return
    columns = 2
    rows = int(np.ceil(len(selected) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(16, 6 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, item in zip(axes, selected):
        plotted = item['result'].plot()
        for true_class, true_box in item['ground_truth']:
            x1, y1, x2, y2 = (int(value) for value in true_box)
            cv2.rectangle(plotted, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = CLASS_NAMES[true_class] if 0 <= true_class < len(CLASS_NAMES) else f'class_{true_class}'
            cv2.putText(plotted, f'GT {label}', (max(0, x1), max(18, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
        axis.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{Path(item['image_path']).name} | FP={item['false_positive']} FN={item['false_negative']} low-conf={item['low_confidence_matches']}")
        axis.axis('off')
    for axis in axes[len(selected):]:
        axis.axis('off')
    figure.suptitle('Contoh prediksi dengan error terbesar', fontsize=16)
    figure.tight_layout()
    plt.show()

display_error_examples(error_items, count=8)

In [ ]:
def size_recall(items):
    rows = []
    for item in items:
        image = cv2.imread(item['image_path'])
        height, width = image.shape[:2]
        predicted_boxes, predicted_classes, _ = prediction_arrays(item['result'])
        for label in load_yolo_labels(item['image_path']):
            true_class = label[0]
            true_box = xywhn_to_xyxy(label, width, height)
            area_ratio = label[3] * label[4]
            bucket = size_bucket(area_ratio)
            matched = any(class_id == true_class and box_iou(true_box, box) >= 0.5 for box, class_id in zip(predicted_boxes, predicted_classes))
            rows.append({'size': bucket, 'matched': matched})
    if not rows:
        return pd.DataFrame()
    result = pd.DataFrame(rows).groupby('size').agg(objects=('matched', 'size'), matched=('matched', 'sum'))
    result['recall_at_iou50'] = result['matched'] / result['objects']
    return result.sort_index()

if error_items:
    print('Recall berdasarkan ukuran objek (prediksi conf=0.001, IoU=0.5):')
    display(size_recall(error_items).round(4))

## 9. Inference gambar dan video

Inference pada gambar test memeriksa perilaku model pada data yang tidak dipakai untuk training. FPS video diukur dengan `perf_counter`; angka ini bergantung pada GPU/CPU, resolusi, model, dan overhead preprocessing/postprocessing. Target 10 FPS harus dicek pada hardware target, bukan hanya di Colab.

In [ ]:
if 'final_model' in globals() and SPLIT_IMAGES['test']:
    sample_paths = random.Random(SEED).sample(SPLIT_IMAGES['test'], min(5, len(SPLIT_IMAGES['test'])))
    prediction_dir = RUNS_ROOT / 'inference' / 'test_images'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    figure, axes = plt.subplots(len(sample_paths), 1, figsize=(14, 5 * len(sample_paths)))
    axes = np.atleast_1d(axes).ravel()
    for axis, image_path in zip(axes, sample_paths):
        result = final_model.predict(source=str(image_path), imgsz=640, conf=0.25, device=DEVICE, verbose=False)[0]
        plotted = result.plot()
        cv2.imwrite(str(prediction_dir / image_path.name), plotted)
        axis.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        axis.set_title(image_path.name)
        axis.axis('off')
    figure.tight_layout()
    plt.show()
else:
    print('Model final atau test split belum tersedia.')

In [ ]:
VIDEO_PATH = PROJECT_ROOT / 'input_video.mp4'
VIDEO_OUTPUT = RUNS_ROOT / 'inference' / 'vehicles_detected.mp4'

# Jika di Colab, upload video lalu ubah VIDEO_PATH ke nama file yang di-upload.
# from google.colab import files
# uploaded_video = files.upload()
# VIDEO_PATH = Path(next(iter(uploaded_video)))

def run_video_inference(video_path, output_path, model):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f'Video tidak dapat dibuka: {video_path}')
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    source_fps = capture.get(cv2.CAP_PROP_FPS) or 0.0
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), source_fps or 30.0, (width, height))
    timings = []
    frame_count = 0
    preview = None
    while True:
        success, frame = capture.read()
        if not success:
            break
        started = time.perf_counter()
        result = model.predict(source=frame, imgsz=640, conf=0.25, device=DEVICE, verbose=False)[0]
        elapsed = time.perf_counter() - started
        timings.append(elapsed)
        annotated = result.plot()
        writer.write(annotated)
        if preview is None:
            preview = annotated.copy()
        frame_count += 1
    capture.release()
    writer.release()
    if not timings:
        raise ValueError('Video tidak memiliki frame.')
    warmup = timings[1:] if len(timings) > 1 else timings
    mean_seconds = float(np.mean(warmup))
    return {
        'frames': frame_count,
        'source_fps': source_fps,
        'inference_fps_mean': 1 / mean_seconds if mean_seconds else float('inf'),
        'inference_ms_mean': mean_seconds * 1000,
        'inference_fps_median': 1 / float(np.median(warmup)) if np.median(warmup) else float('inf'),
        'preview': preview,
    }

if 'final_model' in globals() and VIDEO_PATH.exists():
    VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    video_stats = run_video_inference(VIDEO_PATH, VIDEO_OUTPUT, final_model)
    print({key: value for key, value in video_stats.items() if key != 'preview'})
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(video_stats['preview'], cv2.COLOR_BGR2RGB))
    plt.title(f"Contoh frame | {video_stats['inference_fps_mean']:.2f} FPS inference")
    plt.axis('off')
    plt.show()
    print(f'Video output: {VIDEO_OUTPUT}')
else:
    print(f'Letakkan video pendek di {VIDEO_PATH} lalu jalankan ulang cell ini.')

## 10. Generate report

Report ini merangkum metrik, eksperimen, ukuran objek, dan FPS yang tersedia dari cell yang sudah dijalankan.

In [ ]:
def metric_line(metrics_dict):
    if not metrics_dict:
        return 'Belum tersedia'
    return (f"precision={metrics_dict['precision']:.4f}, recall={metrics_dict['recall']:.4f}, "
            f"F1={metrics_dict['f1']:.4f}, mAP50={metrics_dict['mAP50']:.4f}, "
            f"mAP50-95={metrics_dict['mAP50-95']:.4f}")

report_lines = [
    '# Vehicles YOLO Report',
    '',
    '## Dataset',
    f'- Location: `{DATASET_ROOT}`',
    '- Classes: ' + ', '.join(CLASS_NAMES),
    f'- Images: ' + ', '.join(f'{split}={len(paths)}' for split, paths in SPLIT_IMAGES.items()),
    '',
    '## Hardware dan konfigurasi',
    f'- Device: `{DEVICE}`',
    f'- GPU: `{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}`',
    '- Image size: 640, merged classes: car/bus/truck, epochs: 100, patience: 30',
    '',
    '## Validasi model',
]
if valid_results:
    for result in valid_results:
        report_lines.append(f"- {result['model']}: {metric_line(result)}")
else:
    report_lines.append('- Belum ada hasil validasi.')

report_lines.extend(['', '## Test final', f'- Model: `{FINAL_MODEL_PATH}`'])
report_lines.append(f"- {metric_line(globals().get('final_test_overall'))}")
report_lines.extend(['', '## Analisis', '- Confusion matrix dan PR curve tersimpan di folder `runs/evaluation/`.', '- Error analysis menampilkan sampel FP, FN, dan matched prediction dengan confidence rendah.', '- Recall per ukuran objek dihitung secara approximate pada IoU 0.5; angka ini bukan pengganti AP resmi per ukuran.'])

if 'video_stats' in globals():
    report_lines.extend(['', '## Inference video', f"- Mean inference FPS: {video_stats['inference_fps_mean']:.2f}", f"- Median inference FPS: {video_stats['inference_fps_median']:.2f}", f"- Target 10 FPS tercapai: {video_stats['inference_fps_mean'] >= 10}"])
else:
    report_lines.extend(['', '## Inference video', '- Belum dijalankan; tambahkan video dan jalankan cell inference.'])

report_lines.extend(['', '## Kesimpulan', '- Gunakan model final untuk tahap OpenCV + WebSocket setelah metrik test dan FPS sesuai kebutuhan.', '- Jika kelas bus masih sering tertukar, periksa kualitas label, tambah data kelas tersebut, dan evaluasi threshold confidence/NMS pada video target.'])
REPORT_PATH = REPORTS_ROOT / 'report.md'
REPORT_PATH.write_text('\n'.join(report_lines) + '\n')
print(REPORT_PATH.read_text())